In [1]:
!pip install mlflow boto3

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os

os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow'
os.environ['MLFLOW_TRACKING_USERNAME'] = 'NTsundere'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'd2d97678bd6872fa72b238e9d94b44144e0715d9'

In [4]:
import mlflow

In [5]:
mlflow.set_experiment("Exp 3 - TfIdf Trigram max_features")

2026/07/25 02:48:25 INFO mlflow.tracking.fluent: Experiment with name 'Exp 3 - TfIdf Trigram max_features' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/02678215784d49588e65c09a6e2783b7', creation_time=1784936906511, experiment_id='3', last_update_time=1784936906511, lifecycle_stage='active', name='Exp 3 - TfIdf Trigram max_features', tags={}>

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [7]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [8]:
def run_experiment_tfidf_max_features(max_features):
    ngram_range = (1, 3)  # Trigram setting

    # Step 2: Vectorization using TF-IDF with varying max_features
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"TFIDF_Trigrams_max_features_{max_features}")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, max_features={max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, max_features={max_features}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_{max_features}")

# Step 6: Test various max_features values
max_features_values = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

for max_features in max_features_values:
    run_experiment_tfidf_max_features(max_features)

2026/07/25 02:49:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 02:49:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TFIDF_Trigrams_max_features_1000 at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3/runs/d223fa90123146128230e2c757a9cf6c
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3


2026/07/25 02:50:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 02:51:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TFIDF_Trigrams_max_features_2000 at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3/runs/3ec0a6acab7d4bd39782b1a33a962832
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3


2026/07/25 02:53:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 02:53:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

🏃 View run TFIDF_Trigrams_max_features_8000 at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3/runs/b17d8300742241949c750593643ccefc
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3


2026/07/25 03:09:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 03:09:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TFIDF_Trigrams_max_features_9000 at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3/runs/f07c733331f04e87bbe3b28d0e731c78
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3


2026/07/25 03:10:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 03:10:28 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TFIDF_Trigrams_max_features_10000 at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3/runs/e00282bf9e20407cba5c62b4a53b0330
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/3
